Question

You are given two datasets:

Dataset 1 — Customer Master

customer_id — unique customer ID

customer_name — name of customer

city — customer city

segment — customer segment

Dataset 2 — Transactions

customer_id — customer reference

purchase_date — date of purchase (YYYY-MM-DD)

amount — purchase amount

product_id — purchased product

Task

Write a PySpark program that:

Creates sample DataFrames for both datasets.

Extracts the year from purchase_date.

Determines dynamically:

current year (latest year in data)

previous year (current − 1)

Calculates for each customer:

total purchase amount in current year

total purchase amount in previous year

Joins the result with the Customer Master dataset.

Produces a final output DataFrame containing:

customer_id
customer_name
city
segment
current_year
current_year_amount
previous_year
previous_year_amount


Displays the final result.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, sum, when, max as spark_max, lit

spark = SparkSession.builder.appName("CustomerYearlyPurchase").getOrCreate()

# --------------------------------------------------
# 1. Customer Master Data
# --------------------------------------------------
customer_data = [
    (1, "Alice", "New York", "Premium"),
    (2, "Bob", "Chicago", "Gold"),
    (3, "Charlie", "Dallas", "Silver")
]

customer_cols = ["customer_id", "customer_name", "city", "segment"]

customer_df = spark.createDataFrame(customer_data, customer_cols)


# --------------------------------------------------
# 2. Transaction Data
# --------------------------------------------------
transaction_data = [
    (1, "2023-05-10", 100, "P1"),
    (1, "2024-01-10", 200, "P2"),
    (1, "2024-02-10", 50,  "P3"),
    (2, "2023-03-15", 300, "P1"),
    (2, "2024-07-01", 150, "P2"),
    (3, "2023-08-20", 400, "P4")
]

trans_cols = ["customer_id", "purchase_date", "amount", "product_id"]

trans_df = spark.createDataFrame(transaction_data, trans_cols)


# --------------------------------------------------
# 3. Extract Year
# --------------------------------------------------
trans_df = trans_df.withColumn("year", year("purchase_date"))


# --------------------------------------------------
# 4. Find Current & Previous Year
# --------------------------------------------------
current_year = trans_df.agg(spark_max("year")).collect()[0][0]
previous_year = current_year - 1


# --------------------------------------------------
# 5. Aggregate Purchase Amounts
# --------------------------------------------------
agg_df = trans_df.groupBy("customer_id").agg(
    sum(when(col("year")==current_year, col("amount"))).alias("current_year_amount"),
    sum(when(col("year")==previous_year, col("amount"))).alias("previous_year_amount")
).fillna(0)


# --------------------------------------------------
# 6. Join with Customer Master
# --------------------------------------------------
final_df = customer_df.join(agg_df, "customer_id", "left") \
    .withColumn("current_year", lit(current_year)) \
    .withColumn("previous_year", lit(previous_year)) \
    .fillna(0)


# --------------------------------------------------
# 7. Show Final Output
# --------------------------------------------------
final_df.select(
    "customer_id",
    "customer_name",
    "city",
    "segment",
    "current_year",
    "current_year_amount",
    "previous_year",
    "previous_year_amount"
).show()
